# Ablation Study: Time-Progressive Input × Feature Checkpoint

12 個模型的消融實驗，涵蓋三個 input 軸（VD / Hist / VD+Hist）× 四個 feature 軸（無 / T0 / T5 / T15）：

| 代號 | 名稱 | VD 前30分鐘 | 歷史均速 | 事故特徵 (attention) |
|------|------|:-----------:|:--------:|:--------------------:|
| a | UNET-VD | ✓ | | |
| b | UNET-Hist | | ✓ | |
| c | UNET-VD+Hist | ✓ | ✓ | |
| a.0 | UNET-VD-T0 | ✓ | | T0 (25 feat) |
| a.5 | UNET-VD-T5 | ✓ | | T5 (29 feat) |
| a.15 | UNET-VD-T15 | ✓ | | T15 (32 feat) |
| b.0 | UNET-Hist-T0 | | ✓ | T0 |
| b.5 | UNET-Hist-T5 | | ✓ | T5 |
| b.15 | UNET-Hist-T15 | | ✓ | T15 |
| c.0 | UNET-VD+Hist-T0 | ✓ | ✓ | T0 |
| c.5 | UNET-VD+Hist-T5 | ✓ | ✓ | T5 |
| c.15 | UNET-VD+Hist-T15 | ✓ | ✓ | T15 |

架構：UNet（與 champion 相同 encoder-decoder 結構）。特徵注入方式：multiplicative attention gate（瓶頸層 512ch）。  
Loss：`UpstreamCongestionWeightedLoss`。Eval split：eval_protocol v1（train ≤ 2024-11-30, val ≤ 2025-02-28）。

In [ ]:
import os, sys, datetime
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/陽交大高速公路計畫/main'
else:
    _here = Path(os.getcwd()).resolve()
    for _p in [_here, *_here.parents]:
        if (_p / 'CLAUDE.md').exists():
            PROJECT_ROOT = str(_p)
            break
    else:
        PROJECT_ROOT = str(_here)

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

RUN_ID = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'RUN_ID       = {RUN_ID}')

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'pyarrow', 'tqdm'], check=True)
import torch
print('cuda:', torch.cuda.is_available())

In [ ]:
import json
import warnings
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}, PyTorch: {torch.__version__}')

In [ ]:
DATA_ROOT   = Path(PROJECT_ROOT) / 'data'
OUTPUT_ROOT = Path(PROJECT_ROOT) / 'output' / 'models' / 'ablation_time_progressive'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

NPZ_PATH     = DATA_ROOT / 'accident_samples_5m.npz'
FEATURES_CSV = Path(PROJECT_ROOT) / 'output' / 'feature_engineering' / 'incident_features_33.csv'
META_PARQUET = DATA_ROOT / 'accident_samples_5m_metadata.parquet'
HIST_DIR     = DATA_ROOT / 'historical_speed_profile_5m'
HIST_CACHE   = OUTPUT_ROOT / 'hist_matrices_cache.npy'

for p in [NPZ_PATH, FEATURES_CSV, META_PARQUET, HIST_DIR]:
    assert p.exists(), f'Missing: {p}'
print('All data paths OK')
print(f'Output  -> {OUTPUT_ROOT}')

## Config & Feature Definitions

In [ ]:
SPEED_MAX = 140.0  # normalization constant (km/h)

@dataclass
class Config:
    upstream_km:     int   = 25
    downstream_km:   int   = 10
    time_before_min: int   = 30
    time_after_min:  int   = 145
    spatial_res_km:  int   = 1
    time_res_min:    int   = 5
    epochs:          int   = 30
    batch_size:      int   = 8
    lr:              float = 1e-4
    congestion_kmh:  float = 60.0
    train_end_date:  str   = '2024-11-30'
    val_end_date:    str   = '2025-02-28'
    pad_target:      int   = 64
    min_before_cov:  float = 0.5

    @property
    def n_space(self):  return (self.upstream_km + self.downstream_km) // self.spatial_res_km + 1
    @property
    def n_time(self):   return (self.time_before_min + self.time_after_min) // self.time_res_min + 1
    @property
    def n_before(self): return self.time_before_min // self.time_res_min
    @property
    def n_after(self):  return self.n_time - self.n_before
    @property
    def pad_h(self):    return (self.pad_target - self.n_space) // 2
    @property
    def pad_w(self):    return (self.pad_target - self.n_time) // 2

cfg = Config()
print(f'n_space={cfg.n_space}, n_time={cfg.n_time}, n_before={cfg.n_before}, n_after={cfg.n_after}')
print(f'pad_h={cfg.pad_h}, pad_w={cfg.pad_w}')

# 32 feature columns (same order as reports/v1_champion.json)
FEATURE_COLS = [
    'mileage', 'injuries', 'fatalities', 'num_lanes', 'lane_blockage_rate',
    'low_temp', 'high_humidity', 'strong_wind', 'raining', 'special_accident',
    'direction_south', 'direction_north', 'direction_east', 'direction_west',
    'highway_1', 'highway_3', 'highway_4', 'highway_6',
    'accident_A1', 'accident_A2', 'accident_A3',
    'tunnel', 'bridge', 'lit', 'shoulder',
    'curve_left', 'curve_right', 'curve_straight',
    'pavement_flexible', 'pavement_rigid', 'sharp_curve', 'steep_slope',
]
assert len(FEATURE_COLS) == 32

# Columns zeroed per time-checkpoint (unavailable before that time)
ZERO_COLS = {
    't15': [],
    't5':  ['injuries', 'fatalities', 'lane_blockage_rate'],
    't0':  ['injuries', 'fatalities', 'lane_blockage_rate',
            'special_accident', 'accident_A1', 'accident_A2', 'accident_A3'],
}

def apply_feature_mask(feat: np.ndarray, feature_set: str) -> np.ndarray:
    feat = feat.copy()
    for col in ZERO_COLS[feature_set]:
        feat[..., FEATURE_COLS.index(col)] = 0.0
    return feat

## Data Loading

NPZ 的完整視窗（upstream=50km, before=60min）比訓練用視窗（upstream=25km, before=30min）大，
這裡裁切到訓練視窗大小。歷史均速矩陣第一次執行時會從 profile parquet 算出並 cache。

In [ ]:
# Load NPZ
raw = np.load(str(NPZ_PATH), allow_pickle=False)
npz_matrices = raw['matrix'].astype(np.float32)  # (N, 61, 61) km/h
npz_ids      = raw['accident_ids']
npz_space    = raw['space_offsets_m']             # (61,)  negative=downstream, positive=upstream
npz_time     = raw['time_offsets_min']            # (61,)  negative=pre-accident
print(f'NPZ: {npz_matrices.shape}  space=[{npz_space[0]},{npz_space[-1]}]m  time=[{npz_time[0]},{npz_time[-1]}]min')

meta_df  = pd.read_parquet(str(META_PARQUET)).set_index('accident_id')
meta_df.index = meta_df.index.astype(str)

feat_raw_df = pd.read_csv(str(FEATURES_CSV))
id_col = 'task_id' if 'task_id' in feat_raw_df.columns else feat_raw_df.columns[0]
feat_raw_df = feat_raw_df.set_index(id_col)
feat_raw_df.index = feat_raw_df.index.astype(str)
feat_raw_df.index.name = 'accident_id'
print(f'Features: {feat_raw_df.shape[0]} rows × {feat_raw_df.shape[1]} columns')

In [ ]:
# Crop NPZ to training window
# Spatial: both windows share downstream bound (-10km), upstream shrinks 50→25km
#   space_offsets_m goes from -10000 to +50000; keep first n_space=36 cells (up to +25km)
space_start = 0
space_end   = cfg.n_space  # 36

# Temporal: find -30 min and +145 min in the NPZ time axis
t_start = int(np.where(npz_time == -cfg.time_before_min)[0][0])      # index of -30 min
t_end   = int(np.where(npz_time ==  cfg.time_after_min)[0][0]) + 1   # index after +145 min

matrices_vd = npz_matrices[:, space_start:space_end, t_start:t_end].copy()  # (N, 36, 36)
assert matrices_vd.shape[1:] == (cfg.n_space, cfg.n_time), matrices_vd.shape
print(f'Cropped VD matrices: {matrices_vd.shape}')
print(f'  space: [{npz_space[space_start]}, {npz_space[space_end-1]}] m')
print(f'  time:  [{npz_time[t_start]}, {npz_time[t_end-1]}] min')

In [ ]:
# Pre-compute historical average speed matrices
# Each matrix: (n_space, n_time) in km/h, matching the training window axes.
# Represents the typical (non-accident) speed profile for that location/time-of-day.
from scripts.lib.inference import build_historical_baseline, AccidentInput, make_profile_index
from scripts.lib.data_loaders import read_historical_profile
from scripts.lib.windows import SampleWindow

TRAIN_WINDOW = SampleWindow(
    upstream_km=cfg.upstream_km, downstream_km=cfg.downstream_km,
    time_before_min=cfg.time_before_min, time_after_min=cfg.time_after_min,
    spatial_resolution_km=cfg.spatial_res_km, resolution_min=cfg.time_res_min,
)

if HIST_CACHE.exists():
    matrices_hist = np.load(str(HIST_CACHE))
    print(f'Loaded cached hist matrices: {matrices_hist.shape}')
else:
    print('Pre-computing historical matrices (cached after first run)...')
    N = len(npz_ids)
    matrices_hist = np.full((N, cfg.n_space, cfg.n_time), np.nan, dtype=np.float32)
    raw_profile_cache = {}    # fw -> raw DataFrame (loaded once per freeway)
    profile_index_cache = {}  # (fw, direction) -> pre-built dict index
    for i, acc_id in enumerate(tqdm(npz_ids, desc='hist precompute')):
        sid = str(acc_id)
        if sid not in meta_df.index:
            continue
        row = meta_df.loc[sid]
        fw        = row['freeway']
        direction = row['direction']
        if fw not in raw_profile_cache:
            raw_profile_cache[fw] = read_historical_profile(freeway=fw, data_root=str(DATA_ROOT))
        key = (fw, direction)
        if key not in profile_index_cache:
            profile_index_cache[key] = make_profile_index(raw_profile_cache[fw], fw, direction)
        accident = AccidentInput(
            report_time=pd.Timestamp(row['report_time']).to_pydatetime(),
            freeway=fw, direction=direction, mileage_km=float(row['mileage_km']),
        )
        mat, _ = build_historical_baseline(accident, TRAIN_WINDOW, profile_index=profile_index_cache[key])
        matrices_hist[i] = mat  # km/h (rasterize_window auto-divides speed_x10)
    np.save(str(HIST_CACHE), matrices_hist)
    print(f'Saved: {matrices_hist.shape} -> {HIST_CACHE}')

print(f'Hist coverage: {(~np.isnan(matrices_hist)).mean():.1%}')

In [ ]:
# Align IDs, filter low pre-accident coverage, train/val/test split
keep_idx, keep_ids = [], []
for i, acc_id in enumerate(npz_ids):
    sid = str(acc_id)
    if sid in meta_df.index and sid in feat_raw_df.index:
        keep_idx.append(i)
        keep_ids.append(sid)

keep_idx  = np.array(keep_idx, dtype=np.intp)
keep_ids  = np.array(keep_ids)
print(f'ID matches: {len(keep_idx)} / {len(npz_ids)}')
if len(keep_idx) == 0:
    raise RuntimeError('No matching accident IDs — check meta_df and feat_raw_df are loaded correctly.')

vd_mats   = matrices_vd[keep_idx]
hist_mats = matrices_hist[keep_idx]
feat_arr  = (feat_raw_df.loc[keep_ids, FEATURE_COLS]
             .apply(pd.to_numeric, errors='coerce').fillna(0.0)
             .values.astype(np.float32))

# Drop samples with less than 50% valid pre-accident VD readings
pre   = vd_mats[:, :, :cfg.n_before]
cov   = (~np.isnan(pre)).mean(axis=(1, 2))
ok    = cov >= cfg.min_before_cov
vd_mats, hist_mats, feat_arr, keep_ids = vd_mats[ok], hist_mats[ok], feat_arr[ok], keep_ids[ok]
print(f'Valid samples: {len(keep_ids)}  (dropped {(~ok).sum()} low-coverage)')

# Time-based split (eval_protocol v1)
rts       = pd.to_datetime(meta_df.loc[keep_ids, 'report_time'].values)
train_end = pd.Timestamp(cfg.train_end_date)
val_end   = pd.Timestamp(cfg.val_end_date)
train_idx = np.where(rts <  train_end)[0]
val_idx   = np.where((rts >= train_end) & (rts < val_end))[0]
test_idx  = np.where(rts >= val_end)[0]
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}')

## Dataset, Model, Loss, Metrics, Training Loop

In [ ]:
class AblationDataset(Dataset):
    """
    input_type : 'vd' (2ch), 'hist' (2ch), or 'vd+hist' (4ch)
    feature_set: None (no features), 't0', 't5', or 't15'
    """
    def __init__(self, vd, hist, feat, indices, cfg, input_type='vd', feature_set=None):
        self.vd, self.hist, self.feat = vd[indices], hist[indices], feat[indices]
        self.cfg, self.input_type, self.feature_set = cfg, input_type, feature_set

    def __len__(self): return len(self.vd)

    def _norm(self, mat):
        spd   = np.clip(mat, 0, SPEED_MAX) / SPEED_MAX
        valid = (~np.isnan(mat)).astype(np.float32)
        return np.nan_to_num(spd, nan=0.0).astype(np.float32), valid

    def __getitem__(self, i):
        c = self.cfg
        channels = []

        if 'vd' in self.input_type:
            spd, val = self._norm(self.vd[i])
            spd[:, c.n_before:] = 0.0
            val[:, c.n_before:] = 0.0
            channels += [spd, val]

        if 'hist' in self.input_type:
            spd_h, val_h = self._norm(self.hist[i])
            channels += [spd_h, val_h]

        x_raw = np.stack(channels, axis=0)
        ph, pw = c.pad_h, c.pad_w
        x_pad  = np.pad(x_raw, ((0,0),(ph,ph),(pw,pw)), mode='reflect')

        feat = self.feat[i].copy()
        if self.feature_set is not None:
            feat = apply_feature_mask(feat, self.feature_set)

        tgt, val_tgt = self._norm(self.vd[i])
        return (
            torch.from_numpy(x_pad),
            torch.from_numpy(feat),
            torch.from_numpy(tgt[:, c.n_before:]),
            torch.from_numpy(val_tgt[:, c.n_before:]),
        )


def make_loaders(input_type, feature_set=None, splits=None):
    """splits: (train_idx, val_idx, test_idx) — defaults to time-based split."""
    tr_idx, vl_idx, te_idx = splits if splits is not None else (train_idx, val_idx, test_idx)
    kw = dict(num_workers=2, pin_memory=(DEVICE == 'cuda'))
    def _ds(idx): return AblationDataset(vd_mats, hist_mats, feat_arr, idx, cfg, input_type, feature_set)
    return (
        DataLoader(_ds(tr_idx), batch_size=cfg.batch_size, shuffle=True,  **kw),
        DataLoader(_ds(vl_idx), batch_size=cfg.batch_size, shuffle=False, **kw),
        DataLoader(_ds(te_idx), batch_size=cfg.batch_size, shuffle=False, **kw),
    )

In [ ]:
class AblationUNet(nn.Module):
    """
    in_channels : 2 (VD or Hist) / 4 (VD+Hist)
    feature_dim : 0 (no features) / 32 (with attention gate)
    """
    def __init__(self, in_channels: int, feature_dim: int = 0):
        super().__init__()
        self.feature_dim = feature_dim

        def enc(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 4, stride=2, padding=1),
                nn.BatchNorm2d(oc), nn.LeakyReLU(0.2, inplace=True))

        def dec(ic, oc):
            return nn.Sequential(
                nn.ConvTranspose2d(ic, oc, 4, stride=2, padding=1),
                nn.BatchNorm2d(oc), nn.ReLU(inplace=True))

        self.enc1 = enc(in_channels, 64)   # -> (B, 64,  32, 32)
        self.enc2 = enc(64,  128)          # -> (B, 128, 16, 16)
        self.enc3 = enc(128, 256)          # -> (B, 256,  8,  8)
        self.enc4 = enc(256, 512)          # -> (B, 512,  4,  4)  bottleneck

        if feature_dim > 0:
            self.feat_gate = nn.Linear(feature_dim, 512)

        self.dec4 = dec(512,     256)
        self.dec3 = dec(256+256, 128)
        self.dec2 = dec(128+128, 64)
        self.dec1 = dec(64+64,   64)
        self.final = nn.Sequential(nn.Conv2d(64, 1, 3, 1, 1), nn.Sigmoid())

    def forward(self, x, feat):
        # Encode
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        # Multiplicative attention gate at bottleneck
        if self.feature_dim > 0:
            gate = torch.sigmoid(self.feat_gate(feat))          # (B, 512)
            e4   = e4 * gate.unsqueeze(-1).unsqueeze(-1)        # channel-wise scale

        # Decode with skip connections
        d4 = self.dec4(e4)
        d3 = self.dec3(torch.cat([d4, e3], dim=1))
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        d1 = self.dec1(torch.cat([d2, e1], dim=1))

        out = self.final(d1).squeeze(1)                          # (B, 64, 64)
        ph, pw = cfg.pad_h, cfg.pad_w
        out = out[:, ph:ph+cfg.n_space, pw:pw+cfg.n_time]       # (B, 36, 36) unpad
        return out[:, :, cfg.n_before:]                          # (B, 36, 30) post-accident only


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Sanity-check shapes
with torch.no_grad():
    for in_ch, fd in [(2, 0), (2, 32), (4, 32)]:
        m = AblationUNet(in_ch, fd)
        x = torch.zeros(2, in_ch, 64, 64)
        f = torch.zeros(2, 32)
        y = m(x, f)
        assert y.shape == (2, cfg.n_space, cfg.n_after), y.shape
        print(f'in_ch={in_ch}, feat_dim={fd} -> output {y.shape} | params={count_params(m):,}')

In [ ]:
# ── Loss ──────────────────────────────────────────────────────────────────────
# Upstream zone: accident location (0 km) + all upstream cells (1–25 km)
# = spatial indices n_down: (10 to 35), 26 cells.
# Downstream cells (indices 0–9, −10 to −1 km) contribute 0.

class UpstreamCongestionWeightedLoss(nn.Module):
    def __init__(self, n_downstream, congestion_kmh=60.0):
        super().__init__()
        self.n_down = n_downstream          # first n_down rows = downstream (excluded)
        self.thresh = congestion_kmh / SPEED_MAX

    def forward(self, pred, target, valid):
        w    = torch.zeros_like(target)
        up   = slice(self.n_down, None)     # indices n_down: = accident + upstream
        cong = (target[:, up, :] < self.thresh).float()
        w[:, up, :] = 2.0 + 3.0 * cong     # 2 (free-flow) or 5 (congested)
        l1    = (pred - target).abs() * w * valid
        denom = valid[:, up, :].sum().clamp(min=1)
        return l1.sum() / denom

criterion = UpstreamCongestionWeightedLoss(
    n_downstream=cfg.downstream_km // cfg.spatial_res_km,   # 10
    congestion_kmh=cfg.congestion_kmh,
)

# ── Metrics ───────────────────────────────────────────────────────────────────
FREE_FLOW_KMH = 80.0

def compute_metrics(preds: np.ndarray, targets: np.ndarray, valids: np.ndarray) -> dict:
    """preds/targets/valids: (N, n_space, n_after), normalized [0,1]."""
    n_down = cfg.downstream_km // cfg.spatial_res_km   # 10; upstream zone = [n_down:]
    v      = valids.astype(bool)

    # Cell MAE (km/h) — full spatial extent for reference
    cell_mae = float(np.abs(preds - targets)[v].mean() * SPEED_MAX) if v.any() else float('nan')

    # Congestion F1 — upstream zone only
    thr  = cfg.congestion_kmh / SPEED_MAX
    v_up = v[:, n_down:, :]
    pc   = (preds[:, n_down:, :]  < thr) & v_up
    tc   = (targets[:, n_down:, :] < thr) & v_up
    tp, fp, fn = (pc & tc).sum(), (pc & ~tc).sum(), (~pc & tc).sum()
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)

    # Upstream congestion MAE (km/h)
    up_v   = valids[:, n_down:, :].astype(bool)
    up_mae = float(np.abs(preds[:,n_down:,:] - targets[:,n_down:,:])[up_v].mean() * SPEED_MAX) if up_v.any() else float('nan')

    # Impact time MAE (minutes)
    ff   = FREE_FLOW_KMH / SPEED_MAX
    errs = []
    for i in range(len(preds)):
        vm = valids[i, n_down:, :].astype(bool)
        if not vm.any(): continue
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            p_ts = np.nanmean(np.where(vm, preds[i,n_down:,:],   np.nan), axis=0)
            t_ts = np.nanmean(np.where(vm, targets[i,n_down:,:], np.nan), axis=0)
        def _rec(ts):
            a = np.where(ts >= ff)[0]
            return int(a[0]) if len(a) else len(ts)
        errs.append(abs(_rec(p_ts) - _rec(t_ts)) * cfg.time_res_min)
    impact_mae = round(float(np.mean(errs)), 2) if errs else float('nan')

    return {
        'cell_mae_kmh':                round(cell_mae, 2),
        'congestion_f1':               round(float(f1), 4),
        'upstream_congestion_mae_kmh': round(up_mae,    2),
        'impact_time_mae_min':         impact_mae,
    }

# ── Training function ─────────────────────────────────────────────────────────
def train_model(name, model, train_loader, val_loader):
    model = model.to(DEVICE)
    opt   = Adam(model.parameters(), lr=cfg.lr, betas=(0.5, 0.999))
    sched = CosineAnnealingLR(opt, T_max=cfg.epochs, eta_min=1e-6)

    ckpt_dir  = OUTPUT_ROOT / name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / 'best_model.pth'

    best_val = float('inf')
    history  = []

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        tr_loss = 0.0
        for x, feat, tgt, val in train_loader:
            x, feat, tgt, val = x.to(DEVICE), feat.to(DEVICE), tgt.to(DEVICE), val.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x, feat), tgt, val)
            loss.backward()
            opt.step()
            tr_loss += loss.item()
        sched.step()
        tr_loss /= len(train_loader)

        model.eval()
        vl_loss = 0.0
        with torch.no_grad():
            for x, feat, tgt, val in val_loader:
                x, feat, tgt, val = x.to(DEVICE), feat.to(DEVICE), tgt.to(DEVICE), val.to(DEVICE)
                vl_loss += criterion(model(x, feat), tgt, val).item()
        vl_loss /= len(val_loader)

        if vl_loss < best_val:
            best_val = vl_loss
            torch.save(model.state_dict(), str(ckpt_path))

        history.append({'epoch': epoch, 'train_loss': round(tr_loss,6), 'val_loss': round(vl_loss,6)})
        if epoch % 5 == 0:
            print(f'  [{name}] {epoch:2d}/{cfg.epochs}  train={tr_loss:.4f}  val={vl_loss:.4f}  best={best_val:.4f}')

    model.load_state_dict(torch.load(str(ckpt_path), map_location=DEVICE))
    model.eval()
    all_p, all_t, all_v = [], [], []
    with torch.no_grad():
        for x, feat, tgt, val in val_loader:
            all_p.append(model(x.to(DEVICE), feat.to(DEVICE)).cpu().numpy())
            all_t.append(tgt.numpy())
            all_v.append(val.numpy())
    metrics = compute_metrics(np.concatenate(all_p), np.concatenate(all_t), np.concatenate(all_v))
    metrics['best_val_loss'] = round(best_val, 6)

    result = {'name': name, 'checkpoint': str(ckpt_path), 'val_metrics': metrics, 'history': history}
    (ckpt_dir / 'result.json').write_text(json.dumps(result, indent=2))
    print(f'[{name}]  cell_MAE={metrics["cell_mae_kmh"]} km/h | '
          f'upstream_MAE={metrics["upstream_congestion_mae_kmh"]} km/h | '
          f'impact_MAE={metrics["impact_time_mae_min"]} min | '
          f'F1={metrics["congestion_f1"]}')
    return result

## Training: 12 Models

每個 cell 可獨立重跑。結果存在 `output/models/ablation_time_progressive/<name>/`。

In [ ]:
all_results = {}

In [ ]:
# a  UNET-VD — VD 前30分鐘 only，無事故特徵
name = 'UNET-VD'
tr, vl, te = make_loaders('vd', feature_set=None)
model = AblationUNet(in_channels=2, feature_dim=0)
all_results[name] = train_model(name, model, tr, vl)

In [ ]:
# b  UNET-Hist — 歷史均速 only，無事故特徵
name = 'UNET-Hist'
tr, vl, te = make_loaders('hist', feature_set=None)
model = AblationUNet(in_channels=2, feature_dim=0)
all_results[name] = train_model(name, model, tr, vl)

In [ ]:
# c  UNET-VD+Hist — 兩種 input 合併 baseline，無事故特徵
name = 'UNET-VD+Hist'
tr, vl, te = make_loaders('vd+hist', feature_set=None)
model = AblationUNet(in_channels=4, feature_dim=0)
all_results[name] = train_model(name, model, tr, vl)

In [ ]:
# a.0  UNET-VD-T0 — VD + 事故剛通報時的特徵（25 active cols）
name = 'UNET-VD-T0'
tr, vl, te = make_loaders('vd', feature_set='t0')
model = AblationUNet(in_channels=2, feature_dim=32)
all_results[name] = train_model(name, model, tr, vl)

In [ ]:
# a.5  UNET-VD-T5 — VD + 巡邏車抵達後特徵（29 active cols）
name = 'UNET-VD-T5'
tr, vl, te = make_loaders('vd', feature_set='t5')
model = AblationUNet(in_channels=2, feature_dim=32)
all_results[name] = train_model(name, model, tr, vl)

In [ ]:
# a.15  UNET-VD-T15 — VD + 完整32特徵（現場評估完成）
name = 'UNET-VD-T15'
tr, vl, te = make_loaders('vd', feature_set='t15')
model = AblationUNet(in_channels=2, feature_dim=32)
all_results[name] = train_model(name, model, tr, vl)

In [ ]:
# b.0  UNET-Hist-T0
name = 'UNET-Hist-T0'
tr, vl, te = make_loaders('hist', feature_set='t0')
model = AblationUNet(in_channels=2, feature_dim=32)
all_results[name] = train_model(name, model, tr, vl)

In [ ]:
# b.5  UNET-Hist-T5
name = 'UNET-Hist-T5'
tr, vl, te = make_loaders('hist', feature_set='t5')
model = AblationUNet(in_channels=2, feature_dim=32)
all_results[name] = train_model(name, model, tr, vl)

In [ ]:
# b.15  UNET-Hist-T15
name = 'UNET-Hist-T15'
tr, vl, te = make_loaders('hist', feature_set='t15')
model = AblationUNet(in_channels=2, feature_dim=32)
all_results[name] = train_model(name, model, tr, vl)

In [ ]:
# c.0  UNET-VD+Hist-T0
name = 'UNET-VD+Hist-T0'
tr, vl, te = make_loaders('vd+hist', feature_set='t0')
model = AblationUNet(in_channels=4, feature_dim=32)
all_results[name] = train_model(name, model, tr, vl)

In [ ]:
# c.5  UNET-VD+Hist-T5
name = 'UNET-VD+Hist-T5'
tr, vl, te = make_loaders('vd+hist', feature_set='t5')
model = AblationUNet(in_channels=4, feature_dim=32)
all_results[name] = train_model(name, model, tr, vl)

In [ ]:
# c.15  UNET-VD+Hist-T15
name = 'UNET-VD+Hist-T15'
tr, vl, te = make_loaders('vd+hist', feature_set='t15')
model = AblationUNet(in_channels=4, feature_dim=32)
all_results[name] = train_model(name, model, tr, vl)

## Results Summary

In [ ]:
import pandas as pd

MODEL_ORDER = [
    'UNET-VD', 'UNET-Hist', 'UNET-VD+Hist',
    'UNET-VD-T0', 'UNET-VD-T5', 'UNET-VD-T15',
    'UNET-Hist-T0', 'UNET-Hist-T5', 'UNET-Hist-T15',
    'UNET-VD+Hist-T0', 'UNET-VD+Hist-T5', 'UNET-VD+Hist-T15',
]

rows = []
for name in MODEL_ORDER:
    if name not in all_results:
        continue
    m = all_results[name]['val_metrics']
    rows.append({
        'model':                     name,
        'impact_time_MAE (min)':     m['impact_time_mae_min'],
        'upstream_cong_MAE (km/h)':  m['upstream_congestion_mae_kmh'],
        'cell_MAE (km/h)':           m['cell_mae_kmh'],
        'congestion_F1':             m['congestion_f1'],
        'best_val_loss':             m['best_val_loss'],
    })

df = pd.DataFrame(rows).set_index('model')
print(df.to_string())

# Save summary
df.reset_index().to_csv(str(OUTPUT_ROOT / 'ablation_summary.csv'), index=False)
print(f'\nSaved -> {OUTPUT_ROOT / "ablation_summary.csv"}')

In [ ]:
# Save all results to a single JSON for later analysis
out_json = OUTPUT_ROOT / 'ablation_all_results.json'
with open(str(out_json), 'w', encoding='utf-8') as f:
    json.dump({
        'run_id':   RUN_ID,
        'cfg':      cfg.__dict__,
        'results':  all_results,
    }, f, indent=2, ensure_ascii=False)
print(f'Full results saved -> {out_json}')

## Random Split (seed=42, 70/15/15)

同樣 12 個模型，改用隨機打亂切分以評估時間切法帶來的 distribution shift 影響。
結果存在 `output/models/ablation_time_progressive/rand/` 下。

In [ ]:
# Random 70/15/15 split (fixed seed for reproducibility)
rng  = np.random.default_rng(42)
perm = rng.permutation(len(keep_ids))
n    = len(perm)
n_tr = int(n * 0.70)
n_vl = int(n * 0.15)

train_idx_rand = perm[:n_tr]
val_idx_rand   = perm[n_tr:n_tr + n_vl]
test_idx_rand  = perm[n_tr + n_vl:]
RAND_SPLITS    = (train_idx_rand, val_idx_rand, test_idx_rand)

print(f'Random split  train={len(train_idx_rand)}  val={len(val_idx_rand)}  test={len(test_idx_rand)}')
print(f'Time split    train={len(train_idx)}        val={len(val_idx)}       test={len(test_idx)}')

In [ ]:
# Train all 12 models with random split
RAND_OUT = OUTPUT_ROOT / 'rand'
RAND_OUT.mkdir(parents=True, exist_ok=True)

MODEL_CONFIGS = [
    ('UNET-VD',          'vd',      None,  2, 0),
    ('UNET-Hist',        'hist',    None,  2, 0),
    ('UNET-VD+Hist',     'vd+hist', None,  4, 0),
    ('UNET-VD-T0',       'vd',      't0',  2, 32),
    ('UNET-VD-T5',       'vd',      't5',  2, 32),
    ('UNET-VD-T15',      'vd',      't15', 2, 32),
    ('UNET-Hist-T0',     'hist',    't0',  2, 32),
    ('UNET-Hist-T5',     'hist',    't5',  2, 32),
    ('UNET-Hist-T15',    'hist',    't15', 2, 32),
    ('UNET-VD+Hist-T0',  'vd+hist', 't0',  4, 32),
    ('UNET-VD+Hist-T5',  'vd+hist', 't5',  4, 32),
    ('UNET-VD+Hist-T15', 'vd+hist', 't15', 4, 32),
]

all_results_rand = {}
_orig_output_root = OUTPUT_ROOT

# Temporarily redirect train_model output dir by monkey-patching OUTPUT_ROOT
OUTPUT_ROOT = RAND_OUT

for name, in_type, feat_set, in_ch, feat_dim in MODEL_CONFIGS:
    tr, vl, te = make_loaders(in_type, feature_set=feat_set, splits=RAND_SPLITS)
    model = AblationUNet(in_channels=in_ch, feature_dim=feat_dim)
    all_results_rand[name] = train_model(name, model, tr, vl)

OUTPUT_ROOT = _orig_output_root

# Save
rand_json = RAND_OUT / 'ablation_rand_results.json'
with open(str(rand_json), 'w', encoding='utf-8') as f:
    json.dump({'cfg': cfg.__dict__, 'results': all_results_rand}, f, indent=2, ensure_ascii=False)
print(f'Random-split results saved -> {rand_json}')

In [ ]:
# Side-by-side comparison: time split vs random split
metric_cols = ['impact_time_mae_min', 'upstream_congestion_mae_kmh', 'congestion_f1']

rows = []
for name in MODEL_ORDER:
    if name not in all_results or name not in all_results_rand:
        continue
    mt = all_results[name]['val_metrics']
    mr = all_results_rand[name]['val_metrics']
    rows.append({
        'model': name,
        'impact_MAE_time':   mt['impact_time_mae_min'],
        'impact_MAE_rand':   mr['impact_time_mae_min'],
        'up_MAE_time':       mt['upstream_congestion_mae_kmh'],
        'up_MAE_rand':       mr['upstream_congestion_mae_kmh'],
        'F1_time':           mt['congestion_f1'],
        'F1_rand':           mr['congestion_f1'],
    })

cmp = pd.DataFrame(rows).set_index('model')
print(cmp.to_string())
cmp.reset_index().to_csv(str(OUTPUT_ROOT / 'split_comparison.csv'), index=False)
print(f'\nSaved -> {OUTPUT_ROOT / "split_comparison.csv"}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

names = [n for n in MODEL_ORDER if n in all_results]
ncols = 4
nrows = (len(names) + ncols - 1) // ncols

# Collect global min/max across all train/val losses to force consistent y-axis
all_losses = []
for name in names:
    hist = all_results[name]['history']
    tr = [h['train_loss'] for h in hist]
    vl = [h['val_loss'] for h in hist]
    all_losses.extend(tr + vl)

if len(all_losses) == 0:
    ymin, ymax = 0.0, 1.0
else:
    ymin, ymax = float(np.min(all_losses)), float(np.max(all_losses))
    pad = (ymax - ymin) * 0.08 if ymax > ymin else 0.1
    ymin -= pad
    ymax += pad

yticks = list(np.linspace(ymin, ymax, 5))

fig = plt.figure(figsize=(ncols * 4.5, nrows * 3.2))
gs = gridspec.GridSpec(nrows, ncols, figure=fig, hspace=0.45, wspace=0.35)

for idx, name in enumerate(names):
    hist = all_results[name]['history']
    tr = [h['train_loss'] for h in hist]
    vl = [h['val_loss'] for h in hist]
    epochs = list(range(1, len(tr) + 1))
    best = all_results[name]['val_metrics']['best_val_loss']

    ax = fig.add_subplot(gs[idx // ncols, idx % ncols])
    ax.plot(epochs, tr, label='train', lw=1.4)
    ax.plot(epochs, vl, label='val',   lw=1.4)
    ax.axhline(best, color='gray', ls='--', lw=0.8, alpha=0.7)

    # consistent y-axis
    ax.set_ylim(ymin, ymax)
    ax.set_yticks(yticks)

    # Highlight last-5-epoch trend
    if len(vl) >= 5:
        tail_slope = (vl[-1] - vl[-5]) / 5
        color = '#e74c3c' if tail_slope < -0.0002 else '#27ae60'
        ax.set_title(f'{name}\\nbest={best:.4f}  Δ={tail_slope:+.4f}/epoch', fontsize=8, color=color)
    else:
        ax.set_title(f'{name}\\nbest={best:.4f}', fontsize=8)

    ax.set_xlabel('Epoch', fontsize=7)
    ax.set_ylabel('Loss', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6)

fig.suptitle('Train / Val Loss Curves  (red = val still decreasing, green = converged)', fontsize=11, y=1.01)
plt.savefig(str(OUTPUT_ROOT / 'loss_curves.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Saved -> loss_curves.png')